Our aim is to put the data we extracted into the format for the app with the help of APIs.

In [ ]:
import json
from openai import OpenAI

with open("data.json","r") as f:
    data=json.load(f)

with open("../saves/apisettings.json", "r") as f:
    api_settings= json.load(f)

We'll provide the template and will ask elements of `data` dictionary to be formatted based on our template.

Before that let's split our data into two to save same tokens. For the first group, we just need the name of the region and the steps to follow. And for the second, most of the region walkthroughs are already in string format, but some include numerated lists which require proper formatting for 'index.html'.

In [2]:
walktrough=[]
reginfo=[]
for i in range(0,len(data)):
    info1={
        "region" : data[i]["region"],
        "Steps" : data[i]["Steps"]
    }
    reginfo.append(info1)
    info2={
        "region" : data[i]["region"],
        "region guide" : data[i]["Step Desc"]
    }
    walktrough.append(info2)

In [ ]:
prompt1="""You are an expert HTML generator. You will receive a Python dictionary containing regions and their associated steps. 
Your task is to generate HTML for each region using the following template:

<div style="text-align: left;" class="mainpanel" id="region_name">
    <h2 style="text-align: center;" class="regiontit" id="regionTitle">Region Name</h2>
    <ul>
        <label class="elden_check">
            <input type="checkbox"> <span>`Step Description`</span>
        </label>
        ...
    </ul>
</div>

Requirements:
1. Replace Region Name with the exact region name which.
2. The id of <div> is the exact region name
3. Each step should become a checkbox <label> as shown.
4. The id inside each <label> should start with a short region code (e.g., 'wl' for West Limgrave) followed by a sequential number.
5. The onchange attribute should match the label id.
6. Keep the HTML clean, indented, and ready to insert into a page.

Example input dictionary:

{'region': 'Roundtable Hold',
 'Steps': [{'step': 1, 'text': 'Head up from Stormhill into Stormveil'},
  {'step': 2, 'text': 'Defeat the Boss'},
}"
Output HTML for the example input should be:
<div style="text-align: left;" class="mainpanel" id="Roundtable Hold">
    <h2 style="text-align: center;" class="regiontit" id="regionTitle">Roundtable Hold</h2>
    <ul>
        <label id="rh1" onchange="regitemsave('rh1')" class="elden_check"><input type="checkbox"> <span>Head up from Stormhill into Stormveil</span></label>
        <label id="rh2" onchange="regitemsave('rh2')" class="elden_check"><input type="checkbox"> <span>Defeat the Boss</span></label>
    </ul>
</div>
Now generate the HTML for this dictionary:
""" + f"{reginfo[12:28]}"

client=OpenAI(api_key=api_settings["openai"])
ans1=client.responses.create(
                model="your fav model",
                input=prompt1,
                temperature=0
            )
print(ans1.output_text)

In [ ]:
prompt2="""You are an expert HTML generator. You will receive a Python list containing regions and their associated steps. 
Your task is to generate HTML for each region using the following rules:

1. If you recieve a list and there are no numbered lists, return the whole as a <div> <p id='region name'+' wlk'> string </p> </div>
2. If you recieve a list and there are numbered lists, return the whole as a <div><p id='region name'> string </p></div> with numbered list appropriately embedded into a <ol> tag with <li> tag
Example input:
[" Head to Mistwood and go down Siofra River Well",
['1-First Pillar is by the site of grace.',
'2-Second Pillar is in a corner.']
]
Example output:
<div style="text-align: left; overflow-y: auto;box-sizing: border-box; max-width: 280px;" class="walktrgh_content">
    <p id= Siofra River wlk> Head to Mistwood and go down Siofra River Well.
        <ol>
            <li> Tree Sentinel</li>
            <li> Explore Coastal Cave</li>
        </ol>
    </p>
</div>

Now generate the HTML for this dictionary:
""" + f"{walktrough[25:28]}"

client=OpenAI(api_key=api_settings["openai"])
ans2=client.responses.create(
                model="gpt-5.2",
                input=prompt2,
                temperature=0
            )
print(ans2.output_text)

In [ ]:
region_nm=[]
for i in range(0,len(data)):
    region_nm.append(data[i]["region"])
print(region_nm)

In [ ]:
prompt3="""You are an expert HTML generator. You will receive a Python list containing region names. 
Your task is to generate HTML for each region using the following template:
Example input:
['West Limgrave','Roundtable Hold']
Example output:
<li onclick="switchregion('West Limgrave')" class="lbutton">West Limgrave</li>
<li onclick="switchregion('Roundtable Hold')" class="lbutton">Roundtable Hold</li>

Now generate the HTML for :
""" + f"{region_nm}"

client=OpenAI(api_key=api_settings["openai"])
ans3=client.responses.create(
                model="gpt-5.2",
                input=prompt3,
                temperature=0
            )
print(ans3.output_text)